# Capstone — Content-Day Performance / Decline Risk

This notebook assembles the evidence from Weeks 4–7 into a research-paper-ready record. It must be run only after the upstream notebooks have been executed. It deliberately refuses to invent results when receipts are missing.

**Research question:** using signals available through March 15, which content items are more likely to show a meaningful decline in GSC clicks per measured day during March 16–31?

**Case-study framing:** this is a practical content operations problem: an SEO reviewer has limited time and needs a transparent way to prioritize investigation of pages whose organic search performance may be weakening. The output supports investigation, not automatic content changes.

In [ ]:
from pathlib import Path
import json, pandas as pd
from IPython.display import display
ROOT=Path.cwd()
if not (ROOT/'work').exists(): ROOT=Path('/content/FlyRank_Internship')
OUT=ROOT/'work'/'outputs'; FIG=ROOT/'work'/'figures'
required=['baseline_metrics.json','w05_metrics.json','w07_playbook_metrics.json']
missing=[f for f in required if not (OUT/f).exists()]
assert not missing, f'Run Weeks 4, 5 and 7 first; missing receipts: {missing}'
w4=json.loads((OUT/required[0]).read_text()); w5=json.loads((OUT/required[1]).read_text()); w7=json.loads((OUT/required[2]).read_text())
comparison=pd.DataFrame(w5['comparison']); display(comparison)
print('Baseline base rate:',w4.get('decline_proxy_base_rate'))
print('Grouped test base rate:',w5.get('test_base_rate'))
print('Selected playbook method:',w7.get('selected_method'))

## 1. Data

**Release:** FlyRank ML Internship pseudonymized warehouse release, build `flyrank_pseudonymized_warehouse_release_v20260703`.

**Tables:** `fact_content_daily_performance` for daily GSC observations and `dim_content` for content dates. The fact table grain is report date × client × content. The project uses the March 2026 partition only.

**Feature window:** March 1–15, 2026. **Outcome window:** March 16–31, 2026. Only rows where `gsc_data_available IS TRUE` are treated as measured. Pages require at least 10 measured days in each half, at least 100 early impressions and at least 5 early clicks.

The June sample partition is not used for model development. Pseudonymous identifiers are used only for grouping and local queue management; do not publish them.

## 2. Methodology

**Proxy label:** 1 when clicks per measured day in March 16–31 are at least 20% lower than March 1–15; otherwise 0. This is an observed change proxy, not a human-verified content-quality label.

**Baseline:** Week-4 fixed rule `log1p(early_impressions)`, using only early-window information.

**Models:** regularized Logistic Regression and Random Forest, trained on early-window numeric features. Client IDs are not model features.

**Validation:** grouped holdout by client, with a random-row split included only as a diagnostic. The grouped split is the headline test. Report base rate and Precision@10/20/50, Average Precision and ROC-AUC. This is a single-month client-generalization test, not a multi-month future-period test.

**Leakage controls:** no late-window metrics, decline labels, trend-derived fields or product scores are features. Missing GSC observations are not silently treated as measured zeroes.

## 3. Results (populate only from executed receipts)

The table below is loaded from the Week-5 metrics receipt. Do not manually replace it with estimates. Explain whether the model beats the rule at the review capacity that matters; report mixed trade-offs rather than selecting only a favorable metric.

In [ ]:
display(comparison)
print('Grouped-client base rate:',w5['test_base_rate'])
print('Random-row diagnostic:',w5['random_split_logistic_diagnostic'])
print('Playbook method:',w7['selected_method'])
print('Baseline precision@K:',w4['baseline_precision_at_k'])

## 4. Limitations and honest framing

The label is a two-window decline proxy and may reflect query-mix changes, seasonality, SERP layout, ranking shifts, data coverage, or tracking changes. A client-group holdout reduces client overlap but does not establish future-month performance. No controlled refresh intervention was evaluated; therefore the project cannot claim that refreshing a page causes clicks to recover. The ranked queue is for human investigation only.

## 5. Ranked recommendations

Use the Week-7 aggregate playbook output to describe which review tiers exist and which method was selected. Do not include pseudonymous page IDs, raw URLs, client labels, or private query text. Recommend checking measurement quality, query mix, seasonality, SERP changes and page intent before considering a content edit.

## 6. Reproducibility and data credit

Repository: https://github.com/yaseen20051/FlyRank_Internship

Notebooks: `work/notebooks/w04_baseline_score.ipynb`, `w05_model.ipynb`, `w06_validation_audit.ipynb`, `w07_action_playbook.ipynb`.

Data credit: Built on the [FlyRank ML Internship dataset](https://flyrank.ai). The release is pseudonymized; no raw client names, URLs or private queries are required for this project. Record package versions, random seed and exact executed metrics from the receipts before publication.

## 7. Five-minute demo outline

1. **Question (45 sec):** how can a content reviewer prioritize investigation of pages showing possible organic-search decline?
2. **Data and method (75 sec):** explain the March feature/outcome split, proxy label, early-only baseline, and client-group holdout.
3. **One chart (60 sec):** show the Week-7 aggregate queue-tier chart; explain what each tier means.
4. **One honest result (60 sec):** state the actual held-out Precision@20 comparison and base rate from the executed metrics. If the model does not beat the baseline, say so.
5. **One recommendation (60 sec):** investigate measurement, query mix and SERP changes before a human decides whether a page needs an edit. Close with the limitations: proxy target, one month, no causal evidence.

## Shareable methodology post

I built a leakage-aware content decline-risk ranking workflow using the FlyRank ML Internship pseudonymized warehouse. The project compares a transparent early-signal baseline with Logistic Regression and Random Forest, using a client-group holdout and a clearly defined future-window decline proxy. The goal is not automated content editing: it is to help a human reviewer decide what to investigate first. [Add one verified result and the deployed paper link after execution.]

## Employer-facing summary (3 sentences)

I built a content-day performance and decline-risk ranking workflow using the FlyRank ML Internship pseudonymized warehouse. It compares an early-signal rule baseline with Logistic Regression and Random Forest, evaluated on clients held out from training. The output is a human-reviewed investigation queue, with findings and limitations reported from the executed validation rather than causal claims.